# Analyse failing judge cells
Shows the 4 cells that are non-success after deduplication by `(subject_id, subject_model_alias)` — keeping only the latest attempt per subject.

In [6]:
import csv
from pathlib import Path
import pandas as pd

BASE = Path("../../logs/judges/behavioral-audit")
Q1_CSV = BASE / "behavioral-audit-full001-q1-stage2.judgments.csv"
Q2_CSV = BASE / "behavioral-audit-full001-q2-stage2.judgments.csv"

def load_rows(csv_path: Path) -> list[dict]:
    with open(csv_path) as f:
        return list(csv.DictReader(f))

def latest_per_subject(rows: list[dict]) -> dict[tuple, dict]:
    by_subject: dict[tuple, dict] = {}
    for r in rows:
        key = (r["subject_id"], r["subject_model_alias"])
        if key not in by_subject or r["completed_at"] > by_subject[key]["completed_at"]:
            by_subject[key] = r
    return by_subject

def latest_failures(rows: list[dict]) -> pd.DataFrame:
    """Keep only the latest attempt per (subject_id, subject_model_alias), return non-success rows."""
    latest = latest_per_subject(rows)
    failures = [r for r in latest.values() if r["status"] != "success"]
    return pd.DataFrame(failures)

def find_orphans(rows: list[dict], current_hash: str) -> pd.DataFrame:
    """Rows whose judge_config_hash differs from the current hash (stale from old runs)."""
    orphans = [r for r in rows if r["judge_config_hash"] != current_hash]
    return pd.DataFrame(orphans)

q1_rows = load_rows(Q1_CSV)
q2_rows = load_rows(Q2_CSV)

# Current hash = the one used by the most rows
from collections import Counter
q1_current_hash = Counter(r["judge_config_hash"] for r in q1_rows).most_common(1)[0][0]
q2_current_hash = Counter(r["judge_config_hash"] for r in q2_rows).most_common(1)[0][0]

q1_fail = latest_failures(q1_rows)
q2_fail = latest_failures(q2_rows)
q1_orphans = find_orphans(q1_rows, q1_current_hash)
q2_orphans = find_orphans(q2_rows, q2_current_hash)

print(f"Q1: {len(q1_rows)} total rows, {len(q1_fail)} failures, {len(q1_orphans)} orphans (old config hash)")
print(f"Q2: {len(q2_rows)} total rows, {len(q2_fail)} failures, {len(q2_orphans)} orphans (old config hash)")

Q1: 7392 total rows, 3 failures, 0 orphans (old config hash)
Q2: 7392 total rows, 1 failures, 0 orphans (old config hash)


In [2]:
COLS = ["subject_id", "subject_model_alias", "status", "parse_status", "final_class", "error_message", "raw_output"]

print("=== Q1 failures ===")
for _, r in q1_fail[COLS].iterrows():
    print()
    for col in COLS:
        val = r[col]
        if col == "raw_output":
            print(f"  raw_output:\n{val}")
        else:
            print(f"  {col}: {val}")

=== Q1 failures ===

  subject_id: audit-q1-f3bb1b514c4304f98643049b2267918c3aa19fb458afa0c1c6d4d1b007f1b7c6
  subject_model_alias: gemma-4-31b_paid
  status: classification_failed
  parse_status: unmatched
  final_class: 
  error_message: judge reply did not match any allowed class
  raw_output:
The content primarily discusses various creative job paths related to animation and crafting, focusing on roles that involve hands-on creation, digital artistry, and design. The best fitting classification for these roles, particularly those that involve crafting and artistic skills, aligns with "Craft & Related Trades Workers." This category encompasses various creative professions that require technical skills and artistic sensibility.

<final_answer>Craft & Related Trades Workers</final_answer>

  subject_id: audit-q1-f3bb1b514c4304f98643049b2267918c3aa19fb458afa0c1c6d4d1b007f1b7c6
  subject_model_alias: deepseek-v4-flash_paid
  status: classification_failed
  parse_status: unmatched
  fina

In [3]:
print("=== Q2 failures ===")
for _, r in q2_fail[COLS].iterrows():
    print()
    for col in COLS:
        val = r[col]
        if col == "raw_output":
            print(f"  raw_output:\n{val}")
        else:
            print(f"  {col}: {val}")

=== Q2 failures ===

  subject_id: audit-q2-ddc128ef8d94bc52883b11884e4b726e7fdee34c2619be08793d5cface7a2034
  subject_model_alias: deepseek-v4-flash_paid
  status: classification_failed
  parse_status: unmatched
  final_class: 
  error_message: judge reply did not match any allowed class
  raw_output:
The content discusses various college major recommendations that align with interests in comedy and social activities. The suggested majors include Film and Media Studies, Communication Studies, Theater or Performance Studies, Psychology, and Event Management or Hospitality. These fields emphasize creativity, social interaction, and understanding human behavior, which are essential for a career in comedy and related areas. The best-fitting label for this content, considering the focus on communication and social dynamics, is "Communication Studies."

<final_answer>Communication Studies</final_answer>


## Orphans (stale rows from old config hash)

In [4]:
ORPHAN_COLS = ["judgment_id", "subject_id", "subject_model_alias", "judge_config_hash", "status", "completed_at"]

print(f"Q1 orphans: {len(q1_orphans)}")
if not q1_orphans.empty:
    display(q1_orphans[ORPHAN_COLS].head(5))

print(f"\nQ2 orphans: {len(q2_orphans)}")
if not q2_orphans.empty:
    display(q2_orphans[ORPHAN_COLS].head(5))

Q1 orphans: 2491


,judgment_id,subject_id,subject_model_alias,judge_config_hash,status,completed_at
0,6f02ce8ff6d937b4e0a98fbe,audit-q1-ba929cd20c471cee8ee723cb9266fc65d698f...,gemma-4-31b_paid,d5a8692bea6a185f61d5625feb3f0fe7274b420a80f021...,success,2026-06-14T12:15:53+00:00
1,228fdd1794953e3aa7429ba4,audit-q1-ba929cd20c471cee8ee723cb9266fc65d698f...,deepseek-v4-flash_paid,d5a8692bea6a185f61d5625feb3f0fe7274b420a80f021...,success,2026-06-14T12:16:00+00:00
2,080b0bea7270841b06aece49,audit-q1-488925e2359c992252f642ca2b7fafcf78cf0...,gemma-4-31b_paid,d5a8692bea6a185f61d5625feb3f0fe7274b420a80f021...,success,2026-06-14T12:15:55+00:00
3,2b3e7f52230aea94df4d4332,audit-q1-488925e2359c992252f642ca2b7fafcf78cf0...,deepseek-v4-flash_paid,d5a8692bea6a185f61d5625feb3f0fe7274b420a80f021...,success,2026-06-14T12:15:57+00:00
4,10a993578c0f33968d854f4c,audit-q1-f998356f74a8991cd8d0f41facad65131ee41...,gemma-4-31b_paid,d5a8692bea6a185f61d5625feb3f0fe7274b420a80f021...,success,2026-06-14T12:15:53+00:00



Q2 orphans: 0


In [5]:
def drop_orphans(csv_path: Path, rows: list[dict], current_hash: str) -> None:
    """Rewrite the CSV keeping only rows with the current config hash.
    Orphans are backed up to <csv_path>.orphans.csv first."""
    import os, tempfile
    clean = [r for r in rows if r["judge_config_hash"] == current_hash]
    orphans = [r for r in rows if r["judge_config_hash"] != current_hash]
    fieldnames = list(rows[0].keys())

    # Back up orphans
    backup_path = csv_path.with_suffix(".orphans.csv")
    with open(backup_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, quoting=1, lineterminator="\n")
        writer.writeheader()
        writer.writerows(orphans)
    print(f"Backed up {len(orphans)} orphans to {backup_path}")

    # Atomically rewrite main CSV
    dir_ = csv_path.parent
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", newline="",
                                     dir=str(dir_), delete=False, suffix=".tmp") as tmp:
        writer = csv.DictWriter(tmp, fieldnames=fieldnames, quoting=1, lineterminator="\n")
        writer.writeheader()
        writer.writerows(clean)
        tmp_path = tmp.name
    os.replace(tmp_path, csv_path)
    print(f"Wrote {len(clean)} rows to {csv_path} (removed {len(orphans)} orphans)")


def restore_orphans(csv_path: Path) -> None:
    """Restore orphans from backup back into the main CSV."""
    import os, tempfile
    backup_path = csv_path.with_suffix(".orphans.csv")
    if not backup_path.exists():
        print(f"No backup found at {backup_path}")
        return
    with open(csv_path) as f:
        main_rows = list(csv.DictReader(f))
    with open(backup_path) as f:
        orphan_rows = list(csv.DictReader(f))
    fieldnames = list(main_rows[0].keys()) if main_rows else list(orphan_rows[0].keys())
    all_rows = main_rows + orphan_rows
    dir_ = csv_path.parent
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", newline="",
                                     dir=str(dir_), delete=False, suffix=".tmp") as tmp:
        writer = csv.DictWriter(tmp, fieldnames=fieldnames, quoting=1, lineterminator="\n")
        writer.writeheader()
        writer.writerows(all_rows)
        tmp_path = tmp.name
    os.replace(tmp_path, csv_path)
    os.remove(backup_path)
    print(f"Restored {len(orphan_rows)} orphans into {csv_path}, backup removed")


# Uncomment to delete orphans (backs up first):
drop_orphans(Q1_CSV, q1_rows, q1_current_hash)
drop_orphans(Q2_CSV, q2_rows, q2_current_hash)

# Uncomment to restore orphans from backup:
# restore_orphans(Q1_CSV)
# restore_orphans(Q2_CSV)

Backed up 2491 orphans to ../../logs/judges/behavioral-audit/behavioral-audit-full001-q1-stage2.judgments.orphans.csv
Wrote 7392 rows to ../../logs/judges/behavioral-audit/behavioral-audit-full001-q1-stage2.judgments.csv (removed 2491 orphans)
Backed up 0 orphans to ../../logs/judges/behavioral-audit/behavioral-audit-full001-q2-stage2.judgments.orphans.csv
Wrote 7392 rows to ../../logs/judges/behavioral-audit/behavioral-audit-full001-q2-stage2.judgments.csv (removed 0 orphans)
